In [34]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report
from scipy.sparse import hstack
import joblib

train_df = pd.read_csv("../data/dataset_treino.csv")
val_df = pd.read_csv("../data/dataset_validacao.csv")
test_df = pd.read_csv("../data/dataset_teste.csv")

def clean_text(text):
    text = str(text).replace("\n", " ").replace("\t", " ")
    text = " ".join(text.split())
    return text

train_df["Text"] = train_df["Text"].astype(str).str.slice(0, 150)
val_df["Text"] = val_df["Text"].astype(str).str.slice(0, 150)
test_df["Text"] = test_df["Text"].astype(str).str.slice(0, 150)

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train_df["source_name"])
y_val = label_encoder.transform(val_df["source_name"])
y_test = label_encoder.transform(test_df["source_name"])

word_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=40000,
    min_df=2,
    sublinear_tf=True
)

char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2, 6),
    max_features=40000,
    min_df=2,
    sublinear_tf=True
)

Xw_train = word_vectorizer.fit_transform(train_df["Text"])
Xw_val = word_vectorizer.transform(val_df["Text"])
Xw_test = word_vectorizer.transform(test_df["Text"])

Xc_train = char_vectorizer.fit_transform(train_df["Text"])
Xc_val = char_vectorizer.transform(val_df["Text"])
Xc_test = char_vectorizer.transform(test_df["Text"])

X_train = hstack([Xw_train, Xc_train]).astype(np.float32)
X_val = hstack([Xw_val, Xc_val]).astype(np.float32)
X_test = hstack([Xw_test, Xc_test]).astype(np.float32)

class SparseDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        x = self.X[idx].toarray().squeeze()
        y = self.y[idx]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.long)

train_loader = DataLoader(SparseDataset(X_train, y_train), batch_size=64, shuffle=True)
val_loader = DataLoader(SparseDataset(X_val, y_val), batch_size=64, shuffle=False)
test_loader = DataLoader(SparseDataset(X_test, y_test), batch_size=64, shuffle=False)

class StrongMLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.35),

            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.25),

            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.net(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = StrongMLP(X_train.shape[1], len(label_encoder.classes_)).to(device)

class_weights = torch.tensor([1.3, 1.3, 0.8, 1.3, 1.3], dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

best_val_acc = -1
patience = 4
patience_counter = 0

for epoch in range(20):
    model.train()
    train_losses = []

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

    model.eval()
    val_preds = []
    val_true = []

    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            outputs = model(xb)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_true.extend(yb.numpy())

    val_acc = accuracy_score(val_true, val_preds)
    val_f1 = f1_score(val_true, val_preds, average="macro")

    print(f"Epoch {epoch+1}: loss={np.mean(train_losses):.4f} val_acc={val_acc:.4f} val_f1={val_f1:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), "../models/PytorchMLP/strong_mlp.pt")
        joblib.dump(word_vectorizer, "../models/PytorchMLP/word_vectorizer.pkl")
        joblib.dump(char_vectorizer, "../models/PytorchMLP/char_vectorizer.pkl")
        joblib.dump(label_encoder, "../models/PytorchMLP/label_encoder.pkl")
        joblib.dump({"input_dim": X_train.shape[1], "num_classes": len(label_encoder.classes_)}, "../models/PytorchMLP/model_meta.pkl")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping.")
            break

model.load_state_dict(torch.load("../models/PytorchMLP/strong_mlp.pt", map_location=device, weights_only=False))
model.eval()

test_preds = []
test_true = []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        outputs = model(xb)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        test_preds.extend(preds)
        test_true.extend(yb.numpy())

print("Test Accuracy:", accuracy_score(test_true, test_preds))
print("Test Macro F1:", f1_score(test_true, test_preds, average="macro"))
print(classification_report(test_true, test_preds, target_names=label_encoder.classes_))

Epoch 1: loss=0.5338 val_acc=0.8971 val_f1=0.8732
Epoch 2: loss=0.0984 val_acc=0.9031 val_f1=0.8802
Epoch 3: loss=0.0228 val_acc=0.9125 val_f1=0.8908
Epoch 4: loss=0.0104 val_acc=0.9104 val_f1=0.8879
Epoch 5: loss=0.0055 val_acc=0.9069 val_f1=0.8829
Epoch 6: loss=0.0035 val_acc=0.9052 val_f1=0.8810
Epoch 7: loss=0.0022 val_acc=0.9074 val_f1=0.8843
Early stopping.
Test Accuracy: 0.8915094339622641
Test Macro F1: 0.8612889305025708
              precision    recall  f1-score   support

   Anthropic       0.83      0.85      0.84       272
      google       0.73      0.71      0.72       285
       human       0.93      0.94      0.94      1137
        meta       0.84      0.89      0.87       276
      openai       0.97      0.92      0.94       362

    accuracy                           0.89      2332
   macro avg       0.86      0.86      0.86      2332
weighted avg       0.89      0.89      0.89      2332



In [35]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from scipy.sparse import hstack
import joblib
import sys

def clean_text(text):
    text = str(text).replace("\n", " ").replace("\t", " ")
    text = " ".join(text.split())
    return text

class StrongMLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.35),

            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.25),

            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.net(x)

def main():

    input_csv = "../data/subm2.csv"
    output_csv = "../subm3/PYTORCHMLP/subm3.csv"

    # carregar dados
    df = pd.read_csv(input_csv, sep=";")
    df["Text"] = df["Text"].apply(clean_text)

    # carregar artefactos
    word_vectorizer = joblib.load("../models/PytorchMLP/word_vectorizer.pkl")
    char_vectorizer = joblib.load("../models/PytorchMLP/char_vectorizer.pkl")
    label_encoder = joblib.load("../models/PytorchMLP/label_encoder.pkl")
    meta = joblib.load("../models/PytorchMLP/model_meta.pkl")

    # transformar texto
    Xw = word_vectorizer.transform(df["Text"])
    Xc = char_vectorizer.transform(df["Text"])
    X = hstack([Xw, Xc]).astype(np.float32)

    # carregar modelo
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = StrongMLP(meta["input_dim"], meta["num_classes"]).to(device)
    model.load_state_dict(torch.load("../models/PytorchMLP/strong_mlp.pt", map_location=device, weights_only=False))
    model.eval()
    # prever
    preds = []
    with torch.no_grad():
        for i in range(X.shape[0]):
            x = torch.tensor(X[i].toarray().squeeze(), dtype=torch.float32).unsqueeze(0).to(device)
            output = model(x)
            pred = torch.argmax(output, dim=1).cpu().item()
            preds.append(pred)

    pred_labels = label_encoder.inverse_transform(preds)

    # guardar submissão
    if "ID" in df.columns:
        submission = pd.DataFrame({
            "ID": df["ID"],
            "Labels": pred_labels
        })
    else:
        submission = pd.DataFrame({
            "Labels": pred_labels
        })

    submission.to_csv(output_csv, sep=";", index=False)
    print(f"Submissão guardada em: {output_csv}")

if __name__ == "__main__":
    main()

Submissão guardada em: ../subm3/PYTORCHMLP/subm3.csv


In [36]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report

pred = pd.read_csv("../subm3/PYTORCHMLP/subm3.csv", sep=";")
true = pd.read_csv("../data/subm2_labels_revealed.csv", sep=";", engine="python")

pred.columns = pred.columns.str.strip()
true.columns = true.columns.str.strip()

df = pred.merge(true, on="ID", how="inner")

y_pred = df["Labels"].astype(str).str.lower().str.strip()
y_true = df["Label"].astype(str).str.lower().str.strip()

print("N linhas comparadas:", len(df))
print("Accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred))

N linhas comparadas: 100
Accuracy: 0.41
              precision    recall  f1-score   support

   anthropic       0.32      0.44      0.37        16
      google       0.00      0.00      0.00        18
       human       0.45      0.91      0.60        34
        meta       0.40      0.12      0.19        16
      openai       0.25      0.06      0.10        16

    accuracy                           0.41       100
   macro avg       0.28      0.31      0.25       100
weighted avg       0.31      0.41      0.31       100



/Users/diogoazevedo/miniconda3/envs/class6dl/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/diogoazevedo/miniconda3/envs/class6dl/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/diogoazevedo/miniconda3/envs/class6dl/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(

In [17]:
print(label_encoder.classes_)

['Anthropic' 'google' 'human' 'meta' 'openai']


In [21]:
print(df.columns)
print(df.head())
print(df["Text"].iloc[0])
print(len(df["Text"].iloc[0]))

NameError: name 'df' is not defined

In [19]:
print(df.head())

NameError: name 'df' is not defined